In [ ]:
pip install pandas numpy


In [ ]:
import pandas as pd
import numpy as np
from numba import njit, jit

# @njit 等价于 @jit(nopython=True)
@jit(nopython=True)
def _consecutive_max_volume_count_core(values: np.ndarray) -> np.ndarray:
    """
    核心计算逻辑，只处理numpy数组
    """
    n = len(values)
    last_greater = np.full(n, -1)
    stack = []
    counts = np.zeros(n, dtype=np.int64)

    for i in range(n):
        while stack and values[stack[-1]] < values[i]:
            stack.pop()
        
        if stack:
            last_greater[i] = stack[-1]
        
        stack.append(i)
        
        if i == 0:
            counts[i] = 0
        else:
            counts[i] = i if last_greater[i] == -1 else (i - (last_greater[i] + 1))
    
    return counts

def consecutive_max_volume_count(s: pd.Series) -> pd.Series:
    """
    计算每个成交量是前面连续多少个成交量的最大值
    :param s: 成交量时间序列，要求索引为DatetimeIndex
    :return: 连续计数序列
    """
    # 调用numba优化的核心函数
    counts = _consecutive_max_volume_count_core(s.values)
    return pd.Series(counts, index=s.index)

# 使用示例保持不变
date_rng = pd.date_range('2023-01-01 09:00', periods=8, freq='T')
df = pd.DataFrame({
    'volume': [6, 1, 2, 4, 3, 5, 1, 5]
}, index=date_rng)

df['consecutive_counts'] = consecutive_max_volume_count(df['volume'])
print(df)


0     10
1     20
2     30
3     40
4     50
5     60
6     70
7     80
8     90
9    100
dtype: int64


In [7]:
# 计算每个值是其前面多少个数据的最大值
# 使用 expanding() 和 cummax() 来高效计算
max_counts = series.expanding().apply(lambda x: (x <= x.iloc[-1]).sum(), raw=True)

# 输出结果
print(max_counts)

AttributeError: 'numpy.ndarray' object has no attribute 'iloc'